# 07 · Biological interpretation
1. **Perturbation similarity network:** perturbations whose delta profiles correlate should target genes in the same complex/pathway.
2. **Quantitative validation:** are highly correlated perturbation pairs enriched for shared GO annotations versus random pairs?
3. **Enrichment** of strongly affected genes for selected perturbations (Enrichr via gseapy; needs internet).

In [ ]:
# ---- Bootstrap: Colab (Google Drive) or local ----
import os, sys, subprocess
REPO_URL = "https://github.com/<your-username>/PerturbAI.git"   # <-- edit once after you create the repo
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    if not os.path.exists("/content/PerturbAI"):
        subprocess.run(["git", "clone", REPO_URL, "/content/PerturbAI"], check=True)
    os.chdir("/content/PerturbAI")
    subprocess.run(["git", "pull", "-q"], check=False)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements-colab.txt"], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", "."], check=True)
else:
    try:
        import perturbai  # installed with `pip install -e .`
    except ImportError:
        sys.path.insert(0, os.path.abspath("../src"))  # fallback when running from notebooks/
from perturbai.paths import get_paths
P = get_paths()
print("Data root:", P.root)

In [ ]:
import numpy as np, pandas as pd, scipy.sparse as sp, matplotlib.pyplot as plt
from perturbai.paths import artifact
from perturbai.utils import set_seed, load_config
CFG = load_config("data_config.yaml"); MCFG = load_config("model_config.yaml")
set_seed(CFG["seed"])
SYNTHETIC = False        # True = 1-minute smoke test on simulated data (no download)
DS = "replogle_k562_essential"
INFO = CFG["datasets"][DS]
PERT, CTRL, BATCH = INFO["perturbation_col"], INFO["control_label"], INFO["batch_col"]
TAG = DS + ("_synthetic" if SYNTHETIC else "")   # artifact prefix

In [ ]:
import gseapy as gp
from scipy.cluster import hierarchy
delta = pd.read_parquet(P.processed / artifact("effects", TAG))
norm = np.linalg.norm(delta.values, axis=1)
top = delta.index[np.argsort(-norm)[: min(150, len(delta))]]
C = np.corrcoef(delta.loc[top].values)
Z = hierarchy.linkage(C, method="average", metric="euclidean")
import seaborn as sns
g = sns.clustermap(pd.DataFrame(C, index=top, columns=top), row_linkage=Z, col_linkage=Z, cmap="coolwarm", center=0,
                   figsize=(9, 9), xticklabels=False, yticklabels=False)
plt.show()
cl = hierarchy.fcluster(Z, t=8, criterion="maxclust")
for k in range(1, 9): print(k, list(top[cl == k][:15]))

In [ ]:
# Do correlated perturbation pairs share biology? (needs internet for the GO library)
lib = gp.get_library("GO_Biological_Process_2023")
gene2terms = {}
for term, genes in lib.items():
    for g in genes: gene2terms.setdefault(g, set()).add(term)
names = list(top); Cm = pd.DataFrame(C, index=names, columns=names)
iu = np.triu_indices(len(names), 1)
pairs = pd.DataFrame({"a": np.array(names)[iu[0]], "b": np.array(names)[iu[1]], "r": Cm.values[iu]})
share = lambda a, b: len(gene2terms.get(a, set()) & gene2terms.get(b, set())) > 0
pairs["shared"] = [share(a, b) for a, b in zip(pairs.a, pairs.b)]
hi = pairs[pairs.r >= pairs.r.quantile(0.99)]
print(f"shared GO term | top 1% correlated pairs: {hi.shared.mean():.2f}  vs all pairs: {pairs.shared.mean():.2f}")

In [ ]:
for p in list(delta.index[np.argsort(-norm)[:3]]):
    d = delta.loc[p]; genes = list(d.index[d <= d.quantile(0.02)])
    try:
        enr = gp.enrichr(gene_list=genes, gene_sets=["GO_Biological_Process_2023", "KEGG_2021_Human"], organism="human", outdir=None)
        print(p, "downregulated genes enrichment"); display(enr.results.sort_values("Adjusted P-value").head(6)[["Gene_set", "Term", "Adjusted P-value"]])
    except Exception as e:
        print("Enrichr unavailable:", e)

**Report:** which clusters recover known complexes (ribosome, proteasome, mitochondrial translation …)? Is the shared-GO enrichment of correlated pairs convincing? Mention limits: GO is incomplete and biased towards well-studied genes.